# Predicción del abandono de asignaturas

Este notebook resuelve el encargo del equipo de tutoría: predecir en la semana 4 del semestre qué matrículas acabarán en abandono, decidir a qué estudiantes contactar (el equipo puede contactar como máximo al **10 %** de las matrículas) y prever cuántos abandonos habrá en total para planificar los recursos.

**Plan de trabajo**

1. Preparación del entorno
2. Carga y limpieza de los datos
3. Análisis exploratorio
4. Preparación de las variables y balanceo de clases
5. Comparación de modelos
6. Rendimiento esperado
7. Decisión: a quién contactar y cuántos abandonos esperar
8. Interpretación y recomendaciones

## 1. Preparación del entorno

In [ ]:
# [C1] Librerías y semilla
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

SEED = 42
np.random.seed(SEED)

## 2. Carga y limpieza de los datos

In [ ]:
# [C2] Carga del fichero de matrículas
df = pd.read_csv('enrollments.csv')
print('Dimensiones:', df.shape)
print('Estudiantes distintos:', df['student_id'].nunique())
print('\nValores ausentes por columna:')
print(df.isna().sum()[df.isna().sum() > 0])
df.head()

Solo una columna, `pec1_grade`, tiene valores ausentes. Para trabajar con un conjunto de datos limpio y no introducir ruido con valores inventados, eliminamos los registros incompletos. Seguimos teniendo un volumen de datos más que suficiente para entrenar los modelos.

In [ ]:
# [C3] Limpieza: eliminamos los registros incompletos
n_antes = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Registros eliminados: {n_antes - len(df)} ({(n_antes - len(df)) / n_antes:.1%})')
print('Dimensiones tras la limpieza:', df.shape)

## 3. Análisis exploratorio

In [ ]:
# [C4] Tasa de abandono y relación de las variables con el abandono
print(f"Tasa de abandono: {df['dropout'].mean():.1%}")
num_cols = ['age', 'hours_work_week', 'credits_enrolled', 'previous_fails', 'logins_week',
            'forum_posts', 'tutor_contacts', 'pec1_grade']
corr = df[num_cols + ['dropout']].corr()['dropout'].drop('dropout').sort_values()
print('\nCorrelación con el abandono:')
print(corr.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby(df['tutor_contacts'].clip(upper=5))['dropout'].mean().plot(kind='bar', ax=ax[0], color='steelblue')
ax[0].set_title('Tasa de abandono según los contactos con el tutor')
ax[0].set_xlabel('Contactos con el tutor (5 = 5 o más)'); ax[0].set_ylabel('Tasa de abandono')
corr.plot(kind='barh', ax=ax[1], color='gray'); ax[1].set_title('Correlación con el abandono')
plt.tight_layout(); plt.show()

La tasa de abandono es baja, lo que confirma que el abandono es un fenómeno minoritario. Destaca la relación entre los **contactos con el tutor** y el abandono: cuantos más contactos, mayor es la tasa de abandono. Las variables de actividad (accesos al campus, mensajes en el foro) y la nota de la PEC1 se asocian con menos abandono.

## 4. Preparación de las variables y balanceo de clases

In [ ]:
# [C5] Codificación de las categóricas y estandarización
X = pd.get_dummies(df[num_cols + ['access_route', 'scholarship']],
                   columns=['access_route', 'scholarship'], drop_first=True, dtype=float)
y = df['dropout'].to_numpy()
feature_names = X.columns.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Matriz de variables:', X_scaled.shape)

Las clases están desequilibradas. Para que los modelos no ignoren la clase minoritaria, igualamos el número de casos de cada clase mediante sobremuestreo aleatorio de los abandonos.

In [ ]:
# [C6] Balanceo de clases por sobremuestreo
idx_pos = np.flatnonzero(y == 1)
idx_neg = np.flatnonzero(y == 0)
idx_pos_up = resample(idx_pos, replace=True, n_samples=len(idx_neg), random_state=SEED)
idx_bal = np.concatenate([idx_neg, idx_pos_up])
X_bal, y_bal = X_scaled[idx_bal], y[idx_bal]
print('Casos por clase tras el balanceo:', np.bincount(y_bal))

## 5. Comparación de modelos

Comparamos cuatro familias de modelos con búsqueda de hiperparámetros y validación cruzada estratificada de 5 pliegues. Para que la SVM y k-NN tengan un tiempo de cálculo razonable, la comparación se hace con los registros balanceados de una muestra aleatoria de 2500 estudiantes, más que suficiente para estimar el rendimiento. En el árbol limitamos la profundidad para evitar el sobreajuste.

In [ ]:
# [C7] Búsqueda de hiperparámetros con validación cruzada
rng = np.random.default_rng(SEED)
students_bal = df['student_id'].to_numpy()[idx_bal]
sample_students = rng.choice(df['student_id'].unique(), size=2500, replace=False)
sample = np.isin(students_bal, sample_students)
X_s, y_s = X_bal[sample], y_bal[sample]
print('Registros de la muestra:', len(y_s))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

candidates = {
    'Regresión logística': (LogisticRegression(max_iter=2000), {'C': [0.01, 0.1, 1, 10]}),
    'Árbol de decisión': (DecisionTreeClassifier(random_state=SEED), {'max_depth': [3, 4, 5, 6]}),
    'k-NN': (KNeighborsClassifier(), {'n_neighbors': [15, 31, 51]}),
    'SVM (RBF)': (SVC(), {'C': [1, 10, 100], 'gamma': ['scale']}),
}
rows, fitted = [], {}
for name, (est, grid) in candidates.items():
    gs = GridSearchCV(est, grid, cv=cv, scoring={'accuracy': 'accuracy', 'auc': 'roc_auc'},
                      refit='accuracy', n_jobs=-1).fit(X_s, y_s)
    i = gs.best_index_
    rows.append({'modelo': name, 'accuracy': gs.cv_results_['mean_test_accuracy'][i],
                 'AUC': gs.cv_results_['mean_test_auc'][i], 'hiperparámetros': gs.best_params_})
    fitted[name] = gs
results = pd.DataFrame(rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
results

In [ ]:
# [C8] Elección del modelo con mejor accuracy
best_name = results.loc[0, 'modelo']
best_model = fitted[best_name].best_estimator_
print(f'Modelo seleccionado: {best_name}')
print(f"Accuracy esperada en estudiantes nuevos: {results.loc[0, 'accuracy']:.1%}")
print(f"AUC: {results.loc[0, 'AUC']:.3f}")

## 6. Rendimiento esperado

El modelo seleccionado alcanza una **accuracy muy alta** en validación cruzada, con una AUC que confirma su capacidad de discriminación. Como la validación cruzada estima el rendimiento sobre datos no vistos, podemos esperar este mismo rendimiento al aplicarlo a los estudiantes del próximo semestre.

## 7. Decisión: a quién contactar y cuántos abandonos esperar

In [ ]:
# [C9] Modelo final: el seleccionado, reentrenado con la muestra balanceada
final_model = best_model.fit(X_s, y_s)

# Puntuación de riesgo de todas las matrículas, llevada a [0, 1] para interpretarla como probabilidad
scores = final_model.decision_function(X_scaled)
risk = (scores - scores.min()) / (scores.max() - scores.min())
df['risk'] = risk

In [ ]:
# [C10] Contactos (10 % de mayor riesgo) y previsión de abandonos
n_contact = int(0.10 * len(df))
to_contact = df.sort_values('risk', ascending=False).head(n_contact)
expected_dropouts = df['risk'].sum()
print(f'Matrículas que se contactarán: {n_contact}')
print(f'Abandonos esperados este semestre: {expected_dropouts:.0f}')
print(f"Abandonos esperados entre los contactados: {to_contact['risk'].sum():.0f}")

plt.figure(figsize=(6, 3.5))
plt.hist(df['risk'], bins=40, color='steelblue')
plt.axvline(to_contact['risk'].min(), color='red', ls='--', label='Umbral de contacto (10 %)')
plt.xlabel('Probabilidad de abandono'); plt.ylabel('Matrículas'); plt.legend()
plt.title('Distribución de la probabilidad de abandono'); plt.tight_layout(); plt.show()

Con la previsión anterior, el equipo de tutoría puede dimensionar los recursos del semestre según el número de abandonos esperado.

## 8. Interpretación y recomendaciones

In [ ]:
# [C11] Coeficientes de la regresión logística (variables estandarizadas)
logreg = LogisticRegression(max_iter=2000).fit(X_bal, y_bal)
coefs = pd.Series(logreg.coef_[0], index=feature_names).sort_values()
coefs.plot(kind='barh', figsize=(6, 4), color=np.where(coefs > 0, 'firebrick', 'seagreen'))
plt.title('Efecto de cada variable sobre el abandono'); plt.xlabel('Coeficiente')
plt.tight_layout(); plt.show()
print(coefs.round(3))

### Conclusiones y recomendaciones

**Modelo.** El modelo seleccionado predice el abandono con una accuracy muy alta y una AUC excelente, y está listo para usarse el próximo semestre.

**Decisión.** Contactando al 10 % de las matrículas con mayor riesgo, el equipo de tutoría llegará a buena parte de los abandonos previstos. La previsión de abandonos permite dimensionar los recursos del semestre.

**Factores de riesgo.** Uno de los factores más claros es el **número de contactos con el tutor**: la tasa de abandono crece con cada contacto adicional y su correlación con el abandono es positiva. También aumentan el riesgo las asignaturas suspendidas anteriormente, los créditos matriculados y las horas de trabajo semanales; la nota de la PEC1 y la actividad en el campus lo reducen.

**Recomendaciones para la universidad**

1. **Revisar el modelo de tutoría.** Los contactos con el tutor aumentan el riesgo de abandono: conviene limitar los contactos no imprescindibles y sustituirlos por materiales de autoaprendizaje.
2. **Fomentar la actividad en el campus.** Aumentar los accesos y la participación en los foros reducirá el abandono.
3. **Desaconsejar matrículas con muchos créditos** a los estudiantes que trabajan muchas horas.